In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN samples;

databaseName
accuweather
bakehouse
databricks
healthverity
information_schema
nyctaxi
sec
tpcds_sf1
tpcds_sf1000
tpch


In [0]:
%sql
SHOW TABLES IN samples.nyctaxi;

database,tableName,isTemporary
nyctaxi,trips,false


In [0]:
%sql
SELECT * FROM samples.nyctaxi.trips
LIMIT 10;

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


In [0]:
%sql
DESCRIBE TABLE samples.nyctaxi.trips;

col_name,data_type,comment
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
trip_distance,double,null
fare_amount,double,null
pickup_zip,int,null
dropoff_zip,int,null


In [0]:
%sql
SELECT COUNT(*)
FROM samples.nyctaxi.trips;

COUNT(*)
21932


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dbsf_2305122
COMMENT 'Lab 02 schema';

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
dbsf_2305122
default
information_schema


In [0]:
%sql
USE CATALOG workspace;

USE SCHEMA dbsf_2305122;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.dbsf_2305122.raw_files
COMMENT 'Landing zone for uncleaned source files';

In [0]:
%sql
SHOW VOLUMES IN workspace.dbsf_2305122;

database,volume_name
dbsf_2305122,raw_files


In [0]:
%sql
DESCRIBE VOLUME workspace.dbsf_2305122.raw_files;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_files,workspace,dbsf_2305122,2305122@kiit.ac.in,,MANAGED,Landing zone for uncleaned source files,VOLUME,VOLUME_DB_STORAGE


In [0]:
df = spark.read.csv(
"/Volumes/workspace/dbsf_2305122/raw_files/sales_sample.csv",
header=True,
inferSchema=True)

display(df)

order_id,order_date,region,product,quantity,unit_price
1001,2026-01-05,North,Keyboard,3,45.0
1002,2026-01-05,South,Monitor,1,189.5
1003,2026-01-06,East,Keyboard,5,45.0
1004,2026-01-07,North,Mouse,10,17.25
1005,2026-01-08,West,Monitor,2,189.5
1006,2026-01-09,South,Docking Station,4,120.0
1007,2026-01-10,East,Mouse,7,17.25
1008,2026-01-11,North,Monitor,3,189.5
1009,2026-01-12,West,Keyboard,2,45.0
1010,2026-01-12,South,Mouse,15,17.25


In [0]:
df.printSchema()

print(df.count(),"rows")

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

12 rows


In [0]:
df.write.mode("overwrite").saveAsTable(
"workspace.dbsf_2305122.sales_raw")

In [0]:
%sql
SHOW TABLES IN workspace.dbsf_2305122;

database,tableName,isTemporary
dbsf_2305122,sales_raw,false


In [0]:
%sql
SELECT region,
SUM(quantity * unit_price) AS revenue
FROM workspace.dbsf_2305122.sales_raw
GROUP BY region
ORDER BY revenue DESC;

region,revenue
North,1116.0
South,928.25
West,469.0
East,465.75


In [0]:
%sql
DESCRIBE EXTENDED workspace.dbsf_2305122.sales_raw;

col_name,data_type,comment
order_id,int,null
order_date,date,null
region,string,null
product,string,null
quantity,int,null
unit_price,double,null
,,
# Delta Statistics Columns,,
Column Names,"unit_price, order_date, region, product, quantity, order_id",
Column Selection Method,first-32,


In [0]:
%fs ls /Volumes/workspace/dbsf_2305122/raw_files/

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2305122/raw_files/sales_sample.csv,sales_sample.csv,546,1786254744000
